# 15. References and Mutability

## Learning objectives
By the end of this notebook, you will be able to:
- Explain mutability and distinguish it from object identity and value.
- Recognize common mutable and immutable built-in types.
- Explain aliasing and shared references.
- Predict how mutable objects behave when passed to functions.
- Avoid unintended shared state in classes.
- Connect mutability to shallow and deep copying.

This notebook builds on `09_Object_References_and_Identity.ipynb` and `10_Shallow_and_Deep_Copy.ipynb`. Here, the focus is mutability, aliasing, function arguments, and object attributes.

## 1. Introduction to References and Mutability

A Python variable is a name bound to an object. Multiple names can refer to the same object.

**Mutability** describes whether an object's state can be changed after it is created.

- **Object value/state:** the data represented by an object.
- **Object identity:** the object's individual identity during its lifetime, observable with `id()`.
- **Mutation:** changing an existing mutable object's contents or state.
- **Reassignment:** binding a variable name to a different object.

These distinctions matter because a change made through one reference may be visible through another reference to the same mutable object.

In [ ]:
numbers = [10, 20, 30]
print(numbers)
print(id(numbers))

A list is mutable, so an element can be changed without replacing the list object. The identity can remain the same even though its contents change.

In [ ]:
numbers = [10, 20, 30]
original_id = id(numbers)

numbers[0] = 100

print(numbers)
print(id(numbers) == original_id)

Reassignment is different: the variable name is bound to another object. The original object may still exist if another reference points to it.

In [ ]:
name = "Python"
original_id = id(name)

name = name + " Programming"

print(name)
print(id(name) == original_id)

The string concatenation creates a new string value and rebinds `name` to that result. Do not rely on specific numeric `id()` values; compare identity with `is` when that is the question.

## 2. Mutable and Immutable Objects

Common mutable built-in types include:
- `list`
- `dict`
- `set`

Common immutable built-in types include:
- `int`
- `float`
- `bool`
- `str`
- `tuple` (with an important nuance below)

For immutable objects, an operation that appears to change a value generally creates or returns another object rather than modifying that object's value in place.

### Example 1: Mutable versus immutable objects

In [ ]:
# Mutable object
numbers = [10, 20, 30]
numbers[0] = 100
print(numbers)  # [100, 20, 30]

# Immutable object
name = "Python"
name = name + " Programming"
print(name)  # Python Programming

The list is modified in place. In the string example, concatenation produces a new string, and `name` is reassigned to refer to it.

### The tuple nuance

A tuple cannot have its element references replaced, but it can contain a reference to a mutable object. That nested object can still be mutated.

In [ ]:
record = ("Komal", ["Python", "SQL"])
record[1].append("Pandas")

print(record)

The tuple structure remains fixed, while the list stored inside it changes. “Immutable tuple” does not mean every object reachable from the tuple is immutable.

### Mutation versus reassignment at a glance

| Operation | What happens? |
|---|---|
| `items.append(4)` | Mutates the existing list |
| `items[0] = 99` | Mutates the existing list |
| `items = [7, 8]` | Rebinds `items` to another list |
| `text = text + "!"` | Creates a string result and rebinds `text` |

## 3. Aliasing and Shared References

**Aliasing** occurs when two or more variable names refer to the same object. A mutation through any alias can be observed through the others.

### Example 2: Aliasing

In [ ]:
list_a = [1, 2, 3]
list_b = list_a

list_b.append(4)

print(list_a)          # [1, 2, 3, 4]
print(list_b)          # [1, 2, 3, 4]
print(list_a is list_b)  # True

Both names refer to one list. `list_b = list_a` does not create a second list; it binds another name to the same object.

### Reassignment does not mutate the shared object

In [ ]:
list_a = [1, 2, 3]
list_b = list_a

list_b = [9, 8]

print(list_a)  # [1, 2, 3]
print(list_b)  # [9, 8]
print(list_a is list_b)  # False

Here, `list_b` is rebound to a new list. The original list remains referenced by `list_a`.

### A common shared-reference bug

A function or another part of a program may receive a list and modify it without the caller expecting that change. Shared nested lists can cause similar surprises. When unexpected changes appear, check whether multiple names or objects point to the same mutable value.

In [ ]:
settings = {"theme": "light", "shortcuts": ["copy", "paste"]}
backup_settings = settings

backup_settings["shortcuts"].append("undo")

print(settings)
print(backup_settings)

The dictionary and its nested list are shared. Assigning the dictionary to another name did not make an independent backup.

## 4. Mutability and Function Arguments

When a function is called, its parameter name is bound to the object passed by the caller. If that object is mutable, mutating it through the parameter can be visible to the caller. Rebinding the parameter itself does not rebind the caller's variable.

### Example 3A: Mutating a list inside a function

In [ ]:
def add_item(items):
    items.append("Python")

languages = ["Java", "C++"]
add_item(languages)
print(languages)  # ['Java', 'C++', 'Python']

The parameter `items` and the caller's variable `languages` refer to the same list during the call. `append()` mutates that list.

### Example 3B: Reassigning a local parameter

In [ ]:
def replace_list(items):
    items = ["Python", "Java"]

languages = ["C++", "JavaScript"]
replace_list(languages)
print(languages)  # ['C++', 'JavaScript']

The assignment inside `replace_list` changes the local name `items`. It does not replace the caller's binding for `languages`.

### Passing a dictionary to a function

In [ ]:
def set_status(profile):
    profile["status"] = "active"

user_profile = {"name": "Asha", "status": "pending"}
set_status(user_profile)
print(user_profile)

The function changes the dictionary itself, so the caller sees the updated key.

### Returning a modified object

A function can mutate an object and return it, or build a new object and return that instead. Choose and document the behavior that best fits the function's purpose.

In [ ]:
def add_language(languages, language):
    languages.append(language)
    return languages

skills = ["Python"]
result = add_language(skills, "SQL")

print(skills)
print(result)
print(skills is result)

### Is Python pass-by-value or pass-by-reference?

A useful description is **call by object sharing** (also called *call by assignment*): the parameter receives a binding to the same object passed by the caller. This explains both behaviors:
- Mutating a shared mutable object can be visible to the caller.
- Rebinding a local parameter does not rebind the caller's variable.

This is more precise than saying Python is simply pass-by-value or traditional pass-by-reference.

## 5. Mutability in Classes and Objects

An instance attribute can refer to a mutable object such as a list or dictionary. If two instances are given the same list, they share it even if the attribute names are different.

### Sharing a list between objects

In [ ]:
shared_items = []

class Basket:
    def __init__(self, name, items):
        self.name = name
        self.items = items

basket1 = Basket("A", shared_items)
basket2 = Basket("B", shared_items)

basket1.items.append("Notebook")

print(basket1.items)
print(basket2.items)
print(basket1.items is basket2.items)

Both `items` attributes refer to `shared_items`. If each basket should have independent contents, do not pass the same mutable list to both objects.

### Example 4A: A mutable class attribute

Class attributes are shared through the class unless an instance shadows the attribute with its own attribute. A mutable class-level list is therefore usually unsuitable for per-instance data.

In [ ]:
class Student:
    subjects = []

student1 = Student()
student2 = Student()

student1.subjects.append("Python")

print(student1.subjects)  # ['Python']
print(student2.subjects)  # ['Python']

Both instances access the same class-level list, so the change appears for both students. This is an example of unintended shared state.

### Example 4B: Give each instance its own list

In [ ]:
class Student:
    def __init__(self):
        self.subjects = []

student1 = Student()
student2 = Student()

student1.subjects.append("Python")

print(student1.subjects)  # ['Python']
print(student2.subjects)  # []

Creating `self.subjects` in `__init__` gives each instance a distinct list.

### Modifying an attribute versus replacing it

Calling a mutating method changes the object referenced by the attribute. Assigning a new list to the attribute changes which object that attribute refers to.

In [ ]:
class Employee:
    def __init__(self, name):
        self.name = name
        self.skills = []

employee1 = Employee("Riya")
employee2 = Employee("Arun")

employee1.skills.append("Python")
print(employee1.skills)
print(employee2.skills)

employee1.skills = ["SQL"]
print(employee1.skills)
print(employee2.skills)

In this design, each employee starts with an independent list. Appending modifies one employee's list; assigning a new list replaces only that employee's `skills` reference.

### Safer defaults for mutable attributes

Avoid using a mutable object as a default argument when each call should receive independent data. A default value is created once when the function is defined.

In [ ]:
# Avoid this when each instance needs its own list:
class BadStudent:
    def __init__(self, subjects=[]):
        self.subjects = subjects

# Safer:
class GoodStudent:
    def __init__(self, subjects=None):
        if subjects is None:
            self.subjects = []
        else:
            self.subjects = list(subjects)

`list(subjects)` creates a shallow list copy when a list is provided. If the elements themselves are mutable and must also be independent, a deep copy may be needed.

## 6. Best Practices and Common Mistakes

- Do not assume assignment creates a copy of a list or dictionary.
- When independent state is required, create a fresh mutable object for each owner.
- Use mutation when changing a shared object is intentional; use reassignment when the name or attribute should refer to another object.
- Be careful when passing mutable objects to functions: a function can change the caller-visible object.
- Avoid mutable class attributes for data that belongs to each instance.
- Avoid mutable default arguments for per-call state.
- When debugging, inspect whether references are shared before blaming a particular statement.

### Choosing a copying strategy

This notebook builds on `10_Shallow_and_Deep_Copy.ipynb`:

- **Assignment** creates another reference to the same object.
- **Shallow copy** creates a new outer container but retains references to nested objects.
- **Deep copy** recursively copies nested objects, subject to the behavior of the objects involved.

Use a shallow copy when a separate outer container is enough. Use a deep copy when nested mutable objects must also be independent. Copying has a cost, so make the choice based on the desired ownership and mutation behavior.

In [ ]:
import copy

original = {"topics": ["Python", "SQL"]}

alias = original
shallow = copy.copy(original)
deep = copy.deepcopy(original)

original["topics"].append("Pandas")

print(alias)    # Nested change is visible
print(shallow)  # Nested list is shared
print(deep)     # Nested list remains independent

### A compact debugging example

In [ ]:
def add_tag(tags, tag):
    tags.append(tag)

first = ["data"]
second = first

add_tag(second, "science")

print(first)              # ['data', 'science']
print(second)             # ['data', 'science']
print(first is second)    # True

To avoid the shared change, pass an independent list when that is the intended behavior:

```python
add_tag(first.copy(), "science")
```

That makes a shallow copy, which is sufficient for this example because the list contains strings. For nested mutable structures, revisit the shallow-versus-deep-copy distinction.

## Key Takeaways

- Variables are names bound to objects; assignment does not automatically copy an object.
- Mutable objects such as lists, dictionaries, and sets can change in place.
- Immutable objects such as strings and integers cannot have their values changed in place.
- Aliases refer to the same object, so mutation through one alias is visible through the others.
- Function parameters are local names bound to passed objects; mutation and reassignment behave differently.
- Mutable class attributes can accidentally share state between instances.
- Use independent instance attributes and intentional copying when independent state is required.
- Choose assignment, shallow copying, or deep copying according to the independence your program needs.